In [1]:
# Loaded data

import pandas as pd

df_clean = pd.read_parquet("../data/processed/online_retail_after_cleaning.parquet")

**Revenue and Order Metrics Creation**

**Objective**

Create consistent, reusable revenue and order-level metrics that:
- reflect real transactional behavior
- support both gross and net performance views
- align with BI and SQL analysis needs

**Business Context**

After previous steps:

- cancellations are excluded
- returns are preserved as negative quantities
- customer validity is explicitly flagged

We can now safely define revenue logic.


In [ ]:
# Line-Level revenue

df_clean['line_revenue'] = df_clean['Quantity']*df_clean['Price']

# Gross vs Net Revenue Flag

df_clean['is_positive_sale'] = df_clean['line_revenue'] > 0
df_clean['is_return_revenue'] = df_clean['line_revenue'] < 0

# Why this matters:

# net revenue = sum of all line_revenue
# gross revenue = sum of positive line_revenue only

# Order Identifier Normilization

df_clean['order_id'] = df_clean['Invoice'].astype(str)

# Even though Invoice already exists:

# - standardizing naming avoids confusion later
# - simplifies SQL and Power BI models

# Order-Level Aggregation Preview

order_preview = (
    df_clean.groupby('order_id').agg(
        order_revenue = ('line_revenue','sum'),
        items=('Quantity', 'sum'),
        lines=('StockCode', 'count')
    )
)


# Business validation:

# - confirms realistic order values
# - ensures returns reduce order revenue correctly
# - helps detect anomalies early

# Order Type Classification

order_preview['is_net_negative'] = order_preview['order_revenue'] < 0

# Interpretation:

# - orders dominated by returns

# useful later for:

# - return behavior analysis
# - customer risk signals

# Attach Order Metrics Back to Transactions

df_clean = df_clean.merge(
    order_preview,
    left_on='order_id',
    right_index=True,
    how='left'
)

# Why merge back:
# - keeps single analytical table

# enables:
# - line-level + order-level analysis
# - flexible SQL window functions

# Revenue Consistency Check
df_clean['line_revenue'].sum(), order_preview["order_revenue"].sum()

# These values must match.

# - If they don’t → something is wrong upstream.

# Summary of Metrics Created

# After this step, we have:

# Line-level:
# - line_revenue
# - is_positive_sale
# - is_return_revenue

# Order-level:
# - order_id
# - order_revenue
# - items
# - lines
# - is_net_negative

# These fields are now:
# - SQL-ready
# - BI-ready
# - analytically consistent

Date & Time Fatures

Objective

Create standardized date and time features that enable:
- time series analysis
- customer lyfecycle analysis
- cohort analysis
- SQL window analysis
- Power BI date modeling

Business Context

The dataset contains a timestamp (InvoiceDate) at invoice-line level.

For analytics, we need:

- a clean order date
- consistent time granularity
- reusable time dimensions

We derive features rather than recompute time logic repeatedly later.

In [ ]:
# Ensure Datetime Type

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# This is required for:

# - time-based grouping
# - extracting date parts
# - SQL export consistency

# Create Order Date

df_clean['order_date'] = df_clean['InvoiceDate'].dt.date

# Why:
# - separates date logic from time

# Core Calendar Features
df_clean['year'] = df_clean['InvoiceDate'].dt.year
df_clean['month'] = df_clean['InvoiceDate'].dt.month
df_clean['year_month'] =  df_clean['InvoiceDate'].dt.to_period("M").astype(str)

# Use cases:

# - monthly revenue trends
# - seasonality analysis
# - SQL GROUP BY year_month

# Week Based Features

df_clean['week'] = df_clean['InvoiceDate'].dt.isocalendar().week.astype(int)
df_clean['year_week'] = (
    df_clean['InvoiceDate'].dt.isocalendar().year.astype(str)
    + "-"
    + df_clean["week"].astype(str).str.zfill(2)
)

# Why ISO week:

# - avoids partial week ambiguity
# - useful for operational trend analysis

# Day-of-Week

df_clean["day_of_week"] = df_clean['InvoiceDate'].dt.dayofweek
df_clean["day_name"] = df_clean['InvoiceDate'].dt.day_name()

# Business use cases:

# - identify peak shopping days
# - operational staffing insights
# - marketing timing analysis

# Hour of Day

df_clean['hour'] = df_clean['InvoiceDate'].dt.hour

# This is optional but helpful for:

# - intraday behavior analysis
# - detecting batch or system-generated invoices

# Sanity Check: Time Coverage After Cleaning
df_clean["InvoiceDate"].agg(["min", "max"])

# Feature Completeness Check
df_clean[["order_date", "year", "month", "year_month", "week", "year_week", "day_of_week", "day_name", "hour"]].isna().sum()

# There should be no missing values

# Summary of Time Features Created
# - now dataset supports daily, weekly, monthly trends
# - cohort and retention analysis
# - SQL Window functions
# - Power BI Date dimension modeling

**Customer and Order Level Features**

**Objective**

Create order and customer level features
- describe purchasing behavior
- enable segmentation and ranking
- supports cohort, retention, and repeat analysis

No rows removed, only aggregate and enrich

Business Context

After previous steps:

- cancellations are excluded
- returns are preserved (negative revenue)
- valid customers are identified
- revenue and time features exist

Now we formalize behavioral summaries

In [ ]:
# Order-Level Features

# Order Timing Features

order_dates = (
    df_clean
    .groupby("order_id")
    .agg(
        order_date=("order_date", "min"),
        order_year_month=("year_month", "first")
    )
)

# Why:

# - an order may contain multiple lines
# - order date should be one value

# needed for:

# - order trends
# - customer frequency analysis

# Merge Order Timing Back

df_clean = df_clean.drop(columns=["order_date", "order_year_month"], errors="ignore")

df_clean = df_clean.merge(
    order_dates,
    left_on="order_id",
    right_index=True,
    how="left"
)

# This keeps:

# - line-level detail
# - order-level context in the same table

# Customer-Level Features

# First & Last Purchase Dates

customer_dates = (
    df_clean.groupby('Customer ID').agg(
        first_purchase_date = ("order_date", "min"),
        last_purchase_date = ("order_date", "max")
    )
)

customer_dates["first_purchase_date"] = pd.to_datetime(
    customer_dates["first_purchase_date"]
)

customer_dates["last_purchase_date"] = pd.to_datetime(
    customer_dates["last_purchase_date"]
)

# These essential for:
# - customer lifecycle analysis

# - cohort analysis

# Customer Order and Revenue Metrics
customer_metrics = (
    df_clean.groupby('Customer ID').agg(
        total_orders = ("order_id", "nunique"),
        total_items = ("Quantity", "sum"),
        total_revenue = ("line_revenue", "sum"),
        avg_order_revenue = ("order_revenue", "mean")
    )
)

# Combine Customer Features
customer_features = (
    customer_dates.join(customer_metrics).reset_index()
)

# This table is:
# - one row per customer
# - clean analytical grain

# ready for:

# SQL ranking
# Power BI dimensions
# cohort logic

# Customer Activity Window
customer_features["customer_lifetime_days"] = (customer_features["last_purchase_date"] - customer_features["first_purchase_date"]).dt.days


# Why:
# - distinguishes one-time buyers vs long-term customers
# - used later for retention metrics

# Attach Customer Features back to Transactions

df_clean = df_clean.merge(
    customer_features,
    left_on="Customer ID",
    right_index=True,
    how="left"
)

# This enables:

# - line-level + customer-level analysis
# - advanced SQL window functions
# - flexible BI slicing

# Sanity Check 
customer_features.describe()

# Look for:

# - negative lifetime (should not exist)
# - extreme revenue outliers (expected but noted)
# - realistic order counts

# Summary of Features Created

# Order-level:
# - order_date
# - order_year_month

# Customer-level:
# - first_purchase
# - last_purchase
# - total_orders
# - total_items
# - total_revenue
# - avg_order_value
# - customer_lifetime_days

# These features now support:
# - customer ranking
# - repeat purchase analysis
# - cohort analysis
# - Pareto (80/20) logic
# - Power BI star schema design


**Save Processed Data**

**Objective**

Persist analysis-ready datasets so that:
- cleaning decisions are frozen and reproducible
- SQL analysis and Power BI modeling start from a stable base
- notebooks remain modular and fast

In [6]:
processed_path = "../data/processed"

transactions_cols = [
    "order_id",
    "InvoiceDate",
    "order_date",
    "year",
    "month",
    "year_month",
    "week",
    "day_name",
    "hour",
    "StockCode",
    "Description",
    "Country",
    "Customer ID",
    "has_customer",
    "Quantity",
    "Price",
    "line_revenue",
    "order_revenue",
    "items",
    "lines",
    "is_positive_sale",
    "is_return_revenue"
]

df_clean[transactions_cols].to_csv(
    f"{processed_path}/analysis_ready_transactions.csv",
    index=False
)

customer_features.to_csv(
    f"{processed_path}/analysis_ready_customer_summary.csv",
    index=False
)

df_clean[transactions_cols].to_parquet(
    f"{processed_path}/analysis_ready_transactions.parquet"
)

customer_features.to_parquet(
    f"{processed_path}/analysis_ready_summary.parquet"
)

